# Capitolul 7: Construirea aplicațiilor de chat
## Microsoft Foundry Models API Quickstart

Acest caiet este adaptat din [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) care include caiete ce accesează serviciile [Azure OpenAI](aoai-assignment.ipynb).

> **Notă:** GitHub Models va fi retras la sfârșitul lunii iulie 2026. Acest caiet se adresează acum [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), care oferă același catalog de modele gratuit de încercat și experiența SDK Azure AI Inference.


# Prezentare generală  
"Modelele lingvistice mari sunt funcții care mapează text în text. Având un șir de text de intrare, un model lingvistic mare încearcă să prezică textul care va urma"(1). Acest caiet "pornire rapidă" va introduce utilizatorii în conceptele de nivel înalt ale LLM, cerințele de bază ale pachetului pentru a începe cu AML, o introducere ușoară în designul promptului și mai multe exemple scurte de diferite cazuri de utilizare. 


## Cuprins  

[Prezentare generală](#overview)  
[Cum să folosești Serviciul OpenAI](#how-to-use-openai-service)  
[1. Crearea Serviciului tău OpenAI](#1.-creating-your-openai-service)  
[2. Instalare](#2.-installation)    
[3. Credentiale](#3.-credentials)  

[Cazuri de utilizare](#use-cases)    
[1. Rezumarea textului](#1.-summarize-text)  
[2. Clasificarea textului](#2.-classify-text)  
[3. Generarea de nume noi pentru produse](#3.-generate-new-product-names)  
[4. Antrenarea fină a unui clasificator](#4.fine-tune-a-classifier)  

[Referințe](#references)


### Creează primul tău prompt  
Acest exercițiu scurt oferă o introducere de bază pentru trimiterea prompturilor către un model în Microsoft Foundry Models pentru o sarcină simplă de „rezumare”.


**Pași**:  
1. Instalează biblioteca `azure-ai-inference` în mediul tău python, dacă nu ai făcut-o deja.  
2. Încarcă bibliotecile standard de asistență și configurează acreditările pentru Microsoft Foundry Models.  
3. Alege un model pentru sarcina ta  
4. Creează un prompt simplu pentru model  
5. Trimite cererea ta către API-ul modelului!  


### 1. Instalare `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Importați biblioteci ajutătoare și creați instanța credentialelor


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Găsirea modelului potrivit  
Modelele precum GPT-4o și GPT-4o mini pot înțelege și genera limbaj natural și sunt disponibile în catalogul Microsoft Foundry Models alături de modele de la Meta, Mistral, Cohere și Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Proiectarea unui prompt  

"Magia modelelor de limbaj mari constă în faptul că, prin antrenarea pentru a minimiza eroarea de predicție pe cantități vaste de text, modelele ajung să învețe concepte utile pentru aceste predicții. De exemplu, ele învață concepte precum"(1):

* cum să scrii corect
* cum funcționează gramatica
* cum să parafrazezi
* cum să răspunzi la întrebări
* cum să porți o conversație
* cum să scrii în multe limbi
* cum să programezi
* etc.

#### Cum să controlezi un model de limbaj mare  
"Dintre toate inputurile către un model de limbaj mare, de departe cel mai influent este promptul text(1).

Modelele mari de limbaj pot fi stimulate să producă ieșiri în câteva moduri:

Instrucțiune: Spune modelului ce dorești
Completare: Induc modelul să completeze începutul a ceea ce dorești
Demonstrație: Arată modelului ce dorești, fie cu:
Câteva exemple în prompt
Sute sau mii de exemple într-un set de date pentru antrenare de tip fine-tuning"



#### Există trei ghiduri de bază pentru crearea prompturilor:

**Arată și spune**. Fă clar ce dorești, fie prin instrucțiuni, exemple sau o combinație a celor două. Dacă vrei ca modelul să ordoneze o listă de obiecte în ordine alfabetică sau să clasifice un paragraf după sentiment, arată-i că asta dorești.

**Furnizează date de calitate**. Dacă încerci să construiești un clasificator sau să faci modelul să urmeze un tipar, asigură-te că există suficiente exemple. Verifică cu atenție exemplele — modelul este de obicei suficient de inteligent să depisteze greșeli ortografice de bază și să-ți ofere un răspuns, dar poate presupune că acestea sunt intenționate și poate influența răspunsul.

**Verifică setările.** Setările temperature și top_p controlează cât de determinist este modelul în generarea unui răspuns. Dacă ceri un răspuns unde există un singur răspuns corect, ar trebui să setezi aceste valori mai jos. Dacă dorești răspunsuri mai diverse, le poți seta mai sus. Greșeala numărul unu pe care o fac oamenii cu aceste setări este să presupună că sunt controale pentru „inteligență” sau „creativitate”.


Sursa: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Trimite!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Repetă aceeași apelare, cum se compară rezultatele?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Rezumă Textul  
#### Provocare  
Rezumă textul adăugând un 'tl;dr:' la sfârșitul unui pasaj de text. Observă cum modelul înțelege să efectueze un număr de sarcini fără instrucțiuni suplimentare. Poți experimenta cu prompturi mai descriptive decât tl;dr pentru a modifica comportamentul modelului și a personaliza rezumatul primit(3).  

Lucrările recente au demonstrat câștiguri substanțiale în multe sarcini și repere NLP prin pre-antrenarea pe un corpus mare de text, urmată de ajustare fină pentru o sarcină specifică. Deși arhitectura este în general independentă de sarcină, această metodă necesită totuși seturi de date specifice sarcinii pentru ajustare fină, cu mii sau zeci de mii de exemple. În schimb, oamenii pot, în general, să execute o sarcină nouă de limbaj doar din câteva exemple sau din instrucțiuni simple - ceva cu care sistemele NLP actuale încă se confruntă. Aici arătăm că mărirea modelelor de limbaj îmbunătățește considerabil performanța agnostic față de sarcină, în regim de puține exemplare, uneori ajungând chiar la competitivitate cu abordările anterioare de ajustare fină care erau de top.  



Tl;dr  


# Exerciții pentru mai multe cazuri de utilizare  
1. Rezumă textul  
2. Clasifică textul  
3. Generează nume noi de produse


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Clasifică Text  
#### Provocare  
Clasificați elementele în categoriile furnizate în momentul inferenței. În exemplul următor, oferim atât categoriile, cât și textul de clasificat în prompt(*playground_reference). 

Cerere client: Bună, una dintre tastele tastaturii laptopului meu s-a rupt recent și am nevoie de o înlocuire:

Categoria clasificată:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Generate New Product Names
#### Challenge
Create product names from examples words. Here we include in the prompt information about the product we are going to generate names for. We also provide a similar example to show the pattern we wish to receive. We have also set the temperature value high to increase randomness and more innovative responses.

Product description: A home milkshake maker
Seed words: fast, healthy, compact.
Product names: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Product description: A pair of shoes that can fit any foot size.
Seed words: adaptable, fit, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Referințe  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Portalul Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Cele mai bune practici pentru reglarea fină a modelelor GPT pentru clasificarea textului](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Pentru Mai Mult Ajutor  
[Echipa de Comercializare OpenAI](AzureOpenAITeam@microsoft.com) 


# Contribuitori
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
